In [1]:
import sys
import os

# 1. Force install delta-spark into the active notebook kernel
print(f"Using Python executable: {sys.executable}")
!{sys.executable} -m pip install delta-spark==4.4.1

# 2. Clean initialization without manual jar hacks
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName("VerifySilverDeltaFixed")
    .master("local[2]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()


Using Python executable: /usr/local/bin/python3
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: pip3 install --upgrade pip


:: loading settings :: url = jar:file:/Users/tech/Library/Python/3.12/lib/python/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/tech/.ivy2.5.2/cache
The jars for the packages stored in: /Users/tech/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-216b0642-44cf-4c65-93fd-5d7a7bbbf70c;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.1 in central
	found io.delta#delta-storage;4.4.1 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta-kern

In [16]:
bronze_cmapss_df = spark.read.format("delta").load("/Users/tech/Desktop/Training/nasa-pyspark-pipeline/data/lakehouse/bronze/cmapss_sensors")
bronze_cmapss_df.show()

+-------+----------+---------+---------+---------+--------+--------+--------+--------+--------+--------+--------+--------+--------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+--------------------+--------------------+
|unit_id|time_cycle|setting_1|setting_2|setting_3|sensor_1|sensor_2|sensor_3|sensor_4|sensor_5|sensor_6|sensor_7|sensor_8|sensor_9|sensor_10|sensor_11|sensor_12|sensor_13|sensor_14|sensor_15|sensor_16|sensor_17|sensor_18|sensor_19|sensor_20|sensor_21|    ingest_timestamp|         source_file|
+-------+----------+---------+---------+---------+--------+--------+--------+--------+--------+--------+--------+--------+--------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+--------------------+--------------------+
|      1|         1|   0.0023|   3.0E-4|    100.0|  518.67|  643.02| 1585.29| 1398.21|   14.62|   21.61|   553.9| 2388

In [15]:
cmapss_agg_df=bronze_cmapss_df.select("unit_id", "source_file").orderBy("source_file", "unit_id").distinct()
cmapss_agg_df.show(200,truncate=False)
#cmapss_agg_df = bronze_cmapss_df.withColumn("unit_id").groupBy("unit_id")

#cmapss_agg_df.show()

+-------+-----------------------------------+
|unit_id|source_file                        |
+-------+-----------------------------------+
|83     |data/landing/cmapss/train_FD001.txt|
|46     |data/landing/cmapss/train_FD001.txt|
|71     |data/landing/cmapss/train_FD001.txt|
|55     |data/landing/cmapss/train_FD001.txt|
|29     |data/landing/cmapss/train_FD001.txt|
|75     |data/landing/cmapss/train_FD001.txt|
|99     |data/landing/cmapss/train_FD001.txt|
|6      |data/landing/cmapss/train_FD001.txt|
|9      |data/landing/cmapss/train_FD001.txt|
|7      |data/landing/cmapss/train_FD001.txt|
|89     |data/landing/cmapss/train_FD001.txt|
|90     |data/landing/cmapss/train_FD001.txt|
|34     |data/landing/cmapss/train_FD001.txt|
|26     |data/landing/cmapss/train_FD001.txt|
|15     |data/landing/cmapss/train_FD001.txt|
|33     |data/landing/cmapss/train_FD001.txt|
|65     |data/landing/cmapss/train_FD001.txt|
|62     |data/landing/cmapss/train_FD001.txt|
|25     |data/landing/cmapss/train

In [17]:
silver_cmapss_df = spark.read.format("delta").load("/Users/tech/Desktop/Training/nasa-pyspark-pipeline/data/lakehouse/silver/cmapss_sensors")
silver_cmapss_df.printSchema()

root
 |-- unit_id: integer (nullable = true)
 |-- time_cycle: integer (nullable = true)
 |-- setting_1: float (nullable = true)
 |-- setting_2: float (nullable = true)
 |-- setting_3: float (nullable = true)
 |-- sensor_1: float (nullable = true)
 |-- sensor_2: float (nullable = true)
 |-- sensor_3: float (nullable = true)
 |-- sensor_4: float (nullable = true)
 |-- sensor_5: float (nullable = true)
 |-- sensor_6: float (nullable = true)
 |-- sensor_7: float (nullable = true)
 |-- sensor_8: float (nullable = true)
 |-- sensor_9: float (nullable = true)
 |-- sensor_10: float (nullable = true)
 |-- sensor_11: float (nullable = true)
 |-- sensor_12: float (nullable = true)
 |-- sensor_13: float (nullable = true)
 |-- sensor_14: float (nullable = true)
 |-- sensor_15: float (nullable = true)
 |-- sensor_16: float (nullable = true)
 |-- sensor_17: float (nullable = true)
 |-- sensor_18: float (nullable = true)
 |-- sensor_19: float (nullable = true)
 |-- sensor_20: float (nullable = true)
 

In [18]:
silver_cmapss_df.select("*").show()

+-------+----------+---------+---------+---------+--------+--------+--------+--------+--------+--------+--------+--------+--------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+--------------------+--------------------+--------------------+
|unit_id|time_cycle|setting_1|setting_2|setting_3|sensor_1|sensor_2|sensor_3|sensor_4|sensor_5|sensor_6|sensor_7|sensor_8|sensor_9|sensor_10|sensor_11|sensor_12|sensor_13|sensor_14|sensor_15|sensor_16|sensor_17|sensor_18|sensor_19|sensor_20|sensor_21|    ingest_timestamp|         source_file|        processed_at|
+-------+----------+---------+---------+---------+--------+--------+--------+--------+--------+--------+--------+--------+--------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+--------------------+--------------------+--------------------+
|      1|         2|   0.0019|  -3.0E-4|    100.0|  518

In [22]:
silver_cmapss_df.select("unit_id", "source_file").orderBy( "source_file", "unit_id").distinct().show(200, truncate=False)

+-------+-----------------------------------+
|unit_id|source_file                        |
+-------+-----------------------------------+
|57     |data/landing/cmapss/test_FD001.txt |
|83     |data/landing/cmapss/train_FD001.txt|
|46     |data/landing/cmapss/train_FD001.txt|
|91     |data/landing/cmapss/test_FD001.txt |
|71     |data/landing/cmapss/train_FD001.txt|
|55     |data/landing/cmapss/train_FD001.txt|
|62     |data/landing/cmapss/test_FD001.txt |
|29     |data/landing/cmapss/train_FD001.txt|
|93     |data/landing/cmapss/test_FD001.txt |
|75     |data/landing/cmapss/train_FD001.txt|
|99     |data/landing/cmapss/train_FD001.txt|
|6      |data/landing/cmapss/train_FD001.txt|
|9      |data/landing/cmapss/train_FD001.txt|
|7      |data/landing/cmapss/train_FD001.txt|
|29     |data/landing/cmapss/test_FD001.txt |
|89     |data/landing/cmapss/train_FD001.txt|
|90     |data/landing/cmapss/train_FD001.txt|
|34     |data/landing/cmapss/train_FD001.txt|
|26     |data/landing/cmapss/train

In [8]:
cmapss_gold_df = spark.read.format("delta").load("/Users/tech/Desktop/Training/nasa-pyspark-pipeline/data/lakehouse/gold/diagnostics")
cmapss_gold_df.printSchema()

root
 |-- cycle: long (nullable = true)
 |-- engine_id: long (nullable = true)
 |-- matched_asrs_narrative: string (nullable = true)
 |-- matched_metadata: string (nullable = true)
 |-- query_text: string (nullable = true)
 |-- sensor_egt: double (nullable = true)



In [9]:
cmapss_gold_df.select("*").show(20, truncate=False)

+-----+---------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------+----------------------------------------------------------------------------------------------------------------+-----------------+
|cycle|engine_id|matched_asrs_narrative                                                                                                                                                                                                                                                        

26/11/16 12:43:40 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 1051028 ms exceeds timeout 120000 ms
26/11/16 12:43:41 WARN SparkContext: Killing executors is not supported by current scheduler.
26/11/16 12:43:43 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:85)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:70

In [2]:
cmapss_gold_fact_engine_telemetery_df = spark.read.format("delta").load("/Users/tech/Desktop/Training/nasa-pyspark-pipeline/data/lakehouse/gold/fact_engine_telemetry")
cmapss_gold_fact_engine_telemetery_df.printSchema()

root
 |-- engine_id: integer (nullable = true)
 |-- cycle: integer (nullable = true)
 |-- setting_1: float (nullable = true)
 |-- setting_2: float (nullable = true)
 |-- setting_3: float (nullable = true)
 |-- sensor_1: float (nullable = true)
 |-- sensor_2: float (nullable = true)
 |-- sensor_3: float (nullable = true)
 |-- sensor_4: float (nullable = true)
 |-- sensor_5: float (nullable = true)
 |-- sensor_6: float (nullable = true)
 |-- sensor_7: float (nullable = true)
 |-- sensor_8: float (nullable = true)
 |-- sensor_9: float (nullable = true)
 |-- sensor_10: float (nullable = true)
 |-- sensor_11: float (nullable = true)
 |-- sensor_12: float (nullable = true)
 |-- sensor_13: float (nullable = true)
 |-- sensor_14: float (nullable = true)
 |-- sensor_15: float (nullable = true)
 |-- sensor_16: float (nullable = true)
 |-- sensor_17: float (nullable = true)
 |-- sensor_18: float (nullable = true)
 |-- sensor_19: float (nullable = true)
 |-- sensor_20: float (nullable = true)
 |--

In [3]:
cmapss_gold_fact_engine_telemetery_df.select("*").show(100, truncate=False)

26/11/18 22:10:19 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---------+-----+---------+---------+---------+--------+--------+--------+--------+--------+--------+--------+--------+--------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------------+--------------------+-------------------+---------------------+---------------------+-------------------+--------------------+-------------------+--------------------+---------------------+--------------------+--------------------+-------------------+--------------------+---------------------+--------------------+---------------------+--------------------+---------------------+----------------------+------------------+---------------------+--------------------+---------------------+----------------------+--------------------+---------------------+--------------------+----------------------+----------------------+---+---------+--------------------------+
|engine_id|cycle|setting_1|setting_2|setting_3|sensor_1|sensor_2|sensor_3|

In [29]:
cmapss_gold_fact_engine_telemetery_df.select("engine_id", "cycle", "max_cycle", "rul").orderBy("engine_id").show(200, truncate=False)

+---------+-----+---------+---+
|engine_id|cycle|max_cycle|rul|
+---------+-----+---------+---+
|1        |1    |192      |125|
|1        |2    |192      |125|
|1        |3    |192      |125|
|1        |4    |192      |125|
|1        |5    |192      |125|
|1        |6    |192      |125|
|1        |7    |192      |125|
|1        |8    |192      |125|
|1        |9    |192      |125|
|1        |10   |192      |125|
|1        |11   |192      |125|
|1        |12   |192      |125|
|1        |13   |192      |125|
|1        |14   |192      |125|
|1        |15   |192      |125|
|1        |16   |192      |125|
|1        |17   |192      |125|
|1        |18   |192      |125|
|1        |19   |192      |125|
|1        |20   |192      |125|
|1        |21   |192      |125|
|1        |22   |192      |125|
|1        |23   |192      |125|
|1        |24   |192      |125|
|1        |25   |192      |125|
|1        |26   |192      |125|
|1        |27   |192      |125|
|1        |28   |192      |125|
|1      

In [4]:
df_gold_test=spark.read.format("delta").load("/Users/tech/Desktop/Training/nasa-pyspark-pipeline/data/lakehouse/gold/fact_engine_telemetry_test")
df_gold_test.printSchema()

root
 |-- engine_id: integer (nullable = true)
 |-- cycle: integer (nullable = true)
 |-- setting_1: float (nullable = true)
 |-- setting_2: float (nullable = true)
 |-- setting_3: float (nullable = true)
 |-- sensor_1: float (nullable = true)
 |-- sensor_2: float (nullable = true)
 |-- sensor_3: float (nullable = true)
 |-- sensor_4: float (nullable = true)
 |-- sensor_5: float (nullable = true)
 |-- sensor_6: float (nullable = true)
 |-- sensor_7: float (nullable = true)
 |-- sensor_8: float (nullable = true)
 |-- sensor_9: float (nullable = true)
 |-- sensor_10: float (nullable = true)
 |-- sensor_11: float (nullable = true)
 |-- sensor_12: float (nullable = true)
 |-- sensor_13: float (nullable = true)
 |-- sensor_14: float (nullable = true)
 |-- sensor_15: float (nullable = true)
 |-- sensor_16: float (nullable = true)
 |-- sensor_17: float (nullable = true)
 |-- sensor_18: float (nullable = true)
 |-- sensor_19: float (nullable = true)
 |-- sensor_20: float (nullable = true)
 |--

In [5]:
df_gold_test.select("*").show()

+---------+-----+---------+---------+---------+--------+--------+--------+--------+--------+--------+--------+--------+--------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+-------------------+--------------------+-------------------+--------------------+---------------------+-------------------+--------------------+-------------------+--------------------+---------------------+--------------------+--------------------+-------------------+-------------------+---------------------+-------------------+---------------------+--------------------+--------------------+----------------------+-------------------+---------------------+--------------------+--------------------+----------------------+-------------------+---------------------+--------------------+--------------------+----------------------+---+---------+--------------------+-------------+
|engine_id|cycle|setting_1|setting_2|setting_3|sensor_1|sensor_2|sens

26/11/19 02:40:08 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 933961 ms exceeds timeout 120000 ms
26/11/19 02:40:08 WARN SparkContext: Killing executors is not supported by current scheduler.
26/11/19 02:40:13 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:34)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.stora

In [15]:
df_gold_test.select("max_cycle").distinct().show()

+---------+
|max_cycle|
+---------+
|     NULL|
+---------+



In [11]:
df_rul=spark.read.format("delta").load("/Users/tech/Desktop/Training/nasa-pyspark-pipeline/data/lakehouse/silver/rul")
df_rul.printSchema()

root
 |-- engine_id: integer (nullable = true)
 |-- rul: integer (nullable = true)



26/11/17 12:08:44 WARN DeltaLog: Change in the table id detected while updating snapshot. 
Previous snapshot = Snapshot(path=file:/Users/tech/Desktop/Training/nasa-pyspark-pipeline/data/lakehouse/silver/rul/_delta_log, version=0, metadata=Metadata(1ede6c55-c2af-4023-aecd-37a718c2b03f,null,null,Format(parquet,Map()),{"type":"struct","fields":[{"name":"engine_id","type":"integer","nullable":true,"metadata":{}},{"name":"rul","type":"integer","nullable":true,"metadata":{}}]},List(),Map(),Some(1794888486627)), logSegment=LogSegment(file:/Users/tech/Desktop/Training/nasa-pyspark-pipeline/data/lakehouse/silver/rul/_delta_log,0,ArraySeq(DeprecatedRawLocalFileStatus{path=file:/Users/tech/Desktop/Training/nasa-pyspark-pipeline/data/lakehouse/silver/rul/_delta_log/00000000000000000000.json; isDirectory=false; length=1177; replication=1; blocksize=33554432; modification_time=1794888487534; access_time=1794888487681; owner=; group=; permission=rw-rw-rw-; isSymlink=false; hasAcl=false; isEncrypted=f

In [12]:
df_gold_test.select("engine_id").distinct().orderBy("engine_id").show(200)


+---------+
|engine_id|
+---------+
|        1|
|        2|
|        3|
|        4|
|        5|
|        6|
|        7|
|        8|
|        9|
|       10|
|       11|
|       12|
|       13|
|       14|
|       15|
|       16|
|       17|
|       18|
|       19|
|       20|
|       21|
|       22|
|       23|
|       24|
|       25|
|       26|
|       27|
|       28|
|       29|
|       30|
|       31|
|       32|
|       33|
|       34|
|       35|
|       36|
|       37|
|       38|
|       39|
|       40|
|       41|
|       42|
|       43|
|       44|
|       45|
|       46|
|       47|
|       48|
|       49|
|       50|
|       51|
|       52|
|       53|
|       54|
|       55|
|       56|
|       57|
|       58|
|       59|
|       60|
|       61|
|       62|
|       63|
|       64|
|       65|
|       66|
|       67|
|       68|
|       69|
|       70|
|       71|
|       72|
|       73|
|       74|
|       75|
|       76|
|       77|
|       78|
|       79|
|       80|
|   

In [13]:
df_rul.select("engine_id").distinct().orderBy("engine_id").show(200)

+---------+
|engine_id|
+---------+
|        1|
|        2|
|        3|
|        4|
|        5|
|        6|
|        7|
|        8|
|        9|
|       10|
|       11|
|       12|
|       13|
|       14|
|       15|
|       16|
|       17|
|       18|
|       19|
|       20|
|       21|
|       22|
|       23|
|       24|
|       25|
|       26|
|       27|
|       28|
|       29|
|       30|
|       31|
|       32|
|       33|
|       34|
|       35|
|       36|
|       37|
|       38|
|       39|
|       40|
|       41|
|       42|
|       43|
|       44|
|       45|
|       46|
|       47|
|       48|
|       49|
|       50|
|       51|
|       52|
|       53|
|       54|
|       55|
|       56|
|       57|
|       58|
|       59|
|       60|
|       61|
|       62|
|       63|
|       64|
|       65|
|       66|
|       67|
|       68|
|       69|
|       70|
|       71|
|       72|
|       73|
|       74|
|       75|
|       76|
|       77|
|       78|
|       79|
|       80|
|   

In [ ]:
df_rul.select("*").show(200)

+---------+---+
|engine_id|rul|
+---------+---+
|        1|112|
|        2| 98|
|        3| 69|
|        4| 82|
|        5| 91|
|        6| 93|
|        7| 91|
|        8| 95|
|        9|111|
|       10| 96|
|       11| 97|
|       12|124|
|       13| 95|
|       14|107|
|       15| 83|
|       16| 84|
|       17| 50|
|       18| 28|
|       19| 87|
|       20| 16|
|       21| 57|
|       22|111|
|       23|113|
|       24| 20|
|       25|145|
|       26|119|
|       27| 66|
|       28| 97|
|       29| 90|
|       30|115|
|       31|  8|
|       32| 48|
|       33|106|
|       34|  7|
|       35| 11|
|       36| 19|
|       37| 21|
|       38| 50|
|       39|142|
|       40| 28|
|       41| 18|
|       42| 10|
|       43| 59|
|       44|109|
|       45|114|
|       46| 47|
|       47|135|
|       48| 92|
|       49| 21|
|       50| 79|
|       51|114|
|       52| 29|
|       53| 26|
|       54| 97|
|       55|137|
|       56| 15|
|       57|103|
|       58| 37|
|       59|114|
|       

26/11/17 15:30:41 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 1015205 ms exceeds timeout 120000 ms
26/11/17 15:30:41 WARN SparkContext: Killing executors is not supported by current scheduler.
26/11/17 15:30:46 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:34)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.stor

In [9]:
df_rul_bronze=spark.read.format("delta").load("/Users/tech/Desktop/Training/nasa-pyspark-pipeline/data/lakehouse/bronze/rul")
df_rul_bronze.printSchema()

root
 |-- rul: integer (nullable = true)
 |-- ingest_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



In [10]:
df_rul_bronze.select("*").show()

+----+--------------------+--------------------+
| rul|    ingest_timestamp|         source_file|
+----+--------------------+--------------------+
|NULL|2026-11-16 23:08:...|data/landing/rul/...|
|NULL|2026-11-16 23:08:...|data/landing/rul/...|
|NULL|2026-11-16 23:08:...|data/landing/rul/...|
|NULL|2026-11-16 23:08:...|data/landing/rul/...|
|NULL|2026-11-16 23:08:...|data/landing/rul/...|
|NULL|2026-11-16 23:08:...|data/landing/rul/...|
|NULL|2026-11-16 23:08:...|data/landing/rul/...|
|NULL|2026-11-16 23:08:...|data/landing/rul/...|
|NULL|2026-11-16 23:08:...|data/landing/rul/...|
|NULL|2026-11-16 23:08:...|data/landing/rul/...|
|NULL|2026-11-16 23:08:...|data/landing/rul/...|
|NULL|2026-11-16 23:08:...|data/landing/rul/...|
|NULL|2026-11-16 23:08:...|data/landing/rul/...|
|NULL|2026-11-16 23:08:...|data/landing/rul/...|
|NULL|2026-11-16 23:08:...|data/landing/rul/...|
|NULL|2026-11-16 23:08:...|data/landing/rul/...|
|NULL|2026-11-16 23:08:...|data/landing/rul/...|
|NULL|2026-11-16 23:

In [ ]:
df_gold_test.join(df_rul, "engine_id", "inner").count()
#13096



13096

In [30]:
df_gold_test.join(df_rul, on= "engine_id", how="inner").select(df_rul["rul"]).distinct().orderBy(df_rul["rul"]).show(200)

+----+
| rul|
+----+
|NULL|
+----+



In [7]:
# Alias them to prevent column name collision on 'rul'
df_test_aliased = df_gold_test.alias("gold")
df_gt_aliased = df_rul.alias("rul_truth")

    # 2. Join explicitly using aliases
df_eval_joined = df_test_aliased.join(
        df_gt_aliased, 
        df_test_aliased["engine_id"] == df_gt_aliased["engine_id"], 
        "inner"
).show()

df_final = (
        df_eval_joined
        .groupBy(df_test_aliased["engine_id"])
        .agg(
            F.max(df_test_aliased["cycle"]).alias("max_cycle"),
            # Explicitly grab ground truth RUL from df_gt (silver_rul)
            F.first(df_gt_aliased["rul"]).alias("true_rul"),
            # Grab your model's prediction or test gold 'rul'
            F.max(df_test_aliased["rul"]).alias("predicted_rul") 
        )
    )

+---------+-----+---------+---------+---------+--------+--------+--------+--------+--------+--------+--------+--------+--------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+--------------------+-------------------+--------------------+-------------------+--------------------+-------------------+---------------------+--------------------+---------------------+--------------------+---------------------+--------------------+----+---------+--------------------+---------+----+
|engine_id|cycle|setting_1|setting_2|setting_3|sensor_1|sensor_2|sensor_3|sensor_4|sensor_5|sensor_6|sensor_7|sensor_8|sensor_9|sensor_10|sensor_11|sensor_12|sensor_13|sensor_14|sensor_15|sensor_16|sensor_17|sensor_18|sensor_19|sensor_20|sensor_21|sensor_2_roll_mean_5|sensor_2_roll_std_5|sensor_3_roll_mean_5|sensor_3_roll_std_5|sensor_4_roll_mean_5|sensor_4_roll_std_5|sensor_11_roll_mean_5|sensor_11_roll_std_5|sensor_12_roll_mean_5|sensor_12_ro

AttributeError: 'NoneType' object has no attribute 'groupBy'

In [7]:
cmapss_golddiagnostics_test_df = spark.read.format("delta").load("/Users/tech/Desktop/Training/nasa-pyspark-pipeline/data/lakehouse/gold/diagnostics")
cmapss_golddiagnostics_test_df.printSchema()

root
 |-- cycle: long (nullable = true)
 |-- engine_id: long (nullable = true)
 |-- matched_asrs_narrative: string (nullable = true)
 |-- matched_metadata: string (nullable = true)
 |-- query_text: string (nullable = true)
 |-- sensor_egt: double (nullable = true)



In [2]:


cmapss_gold_telemetery_test_df = spark.read.format("delta").load("/Users/tech/Desktop/Training/nasa-pyspark-pipeline/data/lakehouse/gold/fact_engine_telemetry_test")
cmapss_gold_telemetery_test_df.printSchema()
cmapss_gold_telemetery_test_df.select("*").show(200, truncate=False)

root
 |-- engine_id: integer (nullable = true)
 |-- cycle: integer (nullable = true)
 |-- setting_1: float (nullable = true)
 |-- setting_2: float (nullable = true)
 |-- setting_3: float (nullable = true)
 |-- sensor_1: float (nullable = true)
 |-- sensor_2: float (nullable = true)
 |-- sensor_3: float (nullable = true)
 |-- sensor_4: float (nullable = true)
 |-- sensor_5: float (nullable = true)
 |-- sensor_6: float (nullable = true)
 |-- sensor_7: float (nullable = true)
 |-- sensor_8: float (nullable = true)
 |-- sensor_9: float (nullable = true)
 |-- sensor_10: float (nullable = true)
 |-- sensor_11: float (nullable = true)
 |-- sensor_12: float (nullable = true)
 |-- sensor_13: float (nullable = true)
 |-- sensor_14: float (nullable = true)
 |-- sensor_15: float (nullable = true)
 |-- sensor_16: float (nullable = true)
 |-- sensor_17: float (nullable = true)
 |-- sensor_18: float (nullable = true)
 |-- sensor_19: float (nullable = true)
 |-- sensor_20: float (nullable = true)
 |--

26/11/19 22:00:01 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---------+-----+---------+---------+---------+--------+--------+--------+--------+--------+--------+--------+--------+--------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+-------------------+--------------------+-------------------+---------------------+---------------------+-------------------+--------------------+-------------------+---------------------+---------------------+---------------------+--------------------+-------------------+--------------------+---------------------+--------------------+---------------------+--------------------+----------------------+----------------------+-------------------+---------------------+--------------------+---------------------+----------------------+-------------------+---------------------+---------------------+----------------------+----------------------+---+---------+--------------------------+-------------+
|engine_id|cycle|setting_1|setting_2|setting_3|senso

In [6]:
cmapss_gold_telemetery_test_df.select("*").filter("rul < 20").count()

107

In [8]:
cmapss_gold_telemetery_test_df.select("*").filter("predicted_rul < 20").count()

251

26/11/20 00:10:02 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 262161 ms exceeds timeout 120000 ms
26/11/20 00:10:02 WARN SparkContext: Killing executors is not supported by current scheduler.
26/11/20 00:10:02 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:34)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.stora

In [7]:
diagnostics_df = spark.read.format("delta").load("/Users/tech/Desktop/Training/nasa-pyspark-pipeline/data/lakehouse/gold/diagnostics")
diagnostics_df.printSchema
#diagnostics_df.select("matched_asrs_narrative").distinct().show(20, truncate=False)
diagnostics_df.select("*").show(100, truncate=False)

+-----+---------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------+-------------------------------------------------------------------------------------------+-----------------+
|cycle|engine_id|matched_asrs_narrative                                                                                                                                                                                                                            